In [ ]:
# -------------------------------------------------------------------------
# 0. Clean & Safe Dependencies Installation
# -------------------------------------------------------------------------
!pip install -q transformers torchvision faiss-cpu scikit-learn pandas numpy


In [ ]:
# -------------------------------------------------------------------------
# 1. Model 7 Data Prep: 2D Spatial Pooling RAD-DINO & Zero-Leakage FAISS RAG
#    Preserves 2D Anatomical Layout (16 Rows x 8 Cols = 128 Tokens)
#    Patient-Leakage-Free GroupShuffleSplit & Exemplar Retrieval
# -------------------------------------------------------------------------
import os, sys, json, subprocess, zipfile, gc, time, random, torch, faiss
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import AutoModel
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

# Set seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Data Prep Compute Device: {device}", flush=True)

work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# -------------------------------------------------------------------------
# Step A: Auto-Detect Metadata in /kaggle/input or Download from PhysioNet
# -------------------------------------------------------------------------
PHYSIONET_USER = os.environ.get("PHYSIONET_USER", "")
PHYSIONET_PASS = os.environ.get("PHYSIONET_PASS", "")

def setup_metadata_file(target_name, cand_names, download_url):
    if os.path.exists(target_name):
        print(f"File {target_name} already ready in working dir.", flush=True)
        return
    # 1. First check if present in /kaggle/input (allows offline execution)
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f in cand_names:
                full_src = os.path.join(root, f)
                print(f"Found {f} locally in /kaggle/input ({full_src}). Decompressing/copying...", flush=True)
                if f.endswith(".gz"):
                    subprocess.run(f"gunzip -c '{full_src}' > {target_name}", shell=True, check=True)
                else:
                    subprocess.run(f"cp '{full_src}' {target_name}", shell=True, check=True)
                return
    # 2. If not found locally, download via wget with PhysioNet credentials
    print(f"{target_name} not found in /kaggle/input. Downloading from PhysioNet via wget...", flush=True)
    netrc_path = os.path.expanduser("~/.netrc")
    if not os.path.exists(netrc_path):
        with open(netrc_path, "w", encoding="utf-8") as f:
            f.write("machine physionet.org" + chr(10) + "login " + str(PHYSIONET_USER) + chr(10) + "password " + str(PHYSIONET_PASS) + chr(10))
        os.chmod(netrc_path, 0o600)
    gz_name = target_name + ".gz"
    subprocess.run(f"wget -q --netrc -O {gz_name} {download_url}", shell=True, check=True)
    subprocess.run(f"gunzip -f {gz_name}", shell=True, check=True)

setup_metadata_file("meta.csv", ["mimic-cxr-2.0.0-metadata.csv.gz", "mimic-cxr-2.0.0-metadata.csv", "meta.csv", "metadata.csv"], "https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv.gz")
setup_metadata_file("chexpert.csv", ["mimic-cxr-2.0.0-chexpert.csv.gz", "mimic-cxr-2.0.0-chexpert.csv", "chexpert.csv"], "https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-chexpert.csv.gz")

# -------------------------------------------------------------------------
# Step B: Auto-Discover MIMIC-CXR Images and Impression Files in /kaggle/input
# -------------------------------------------------------------------------
input_base = Path("/kaggle/input")
file_index = {}
impression_file_paths = []
reports_zip_path = None

print("Indexing available MIMIC-CXR files in /kaggle/input...", flush=True)
for root, _, files in os.walk(input_base):
    for f in files:
        full_path = os.path.join(root, f)
        if f in ["mimic_train_impressions.csv", "train_imp.csv", "mimic_test_impressions.csv", "test_imp.csv"]:
            if full_path not in impression_file_paths:
                impression_file_paths.append(full_path)
        elif f == "mimic-cxr-reports.zip":
            reports_zip_path = full_path
        elif f.endswith(".jpg"):
            parts = full_path.split("/")
            rel_path = None
            for prefix in ("p10", "p11"):
                if prefix in parts:
                    idx = parts.index(prefix)
                    rel_path = "/".join(parts[idx:])
                    break
            if rel_path is not None and rel_path not in file_index:
                file_index[rel_path] = full_path

if not impression_file_paths:
    sys.exit("Error: No report impression CSV files found in /kaggle/input/.")

TOTAL_REFERENCE_POOL = 75216
indexed_count = len(file_index)
coverage_pct = (indexed_count / TOTAL_REFERENCE_POOL) * 100.0
print(f"Pool Indexing Report: Found {indexed_count} unique images on disk ({coverage_pct:.1f}% of 75,216 reference pool).", flush=True)

report_dfs = [pd.read_csv(p) for p in impression_file_paths]
report_df = pd.concat(report_dfs, ignore_index=True).drop_duplicates(subset=["dicom_id"])

meta_df = pd.read_csv("meta.csv")
chex_df = pd.read_csv("chexpert.csv")

chex_label_cols = [c for c in chex_df.columns if c not in ["subject_id", "study_id"]]
for c in chex_label_cols:
    chex_df[c] = chex_df[c].fillna(0.0).replace(-1.0, 0.0)

# Filter cohort: p10 and p11
mask_cohort = (meta_df["subject_id"] >= 10000000) & (meta_df["subject_id"] < 12000000)
meta_cohort = meta_df[mask_cohort].copy()

merged_df = pd.merge(meta_cohort, chex_df[["study_id"] + chex_label_cols], on="study_id", how="inner")
merged_df = pd.merge(merged_df, report_df[["dicom_id", "report"]], on="dicom_id", how="left")
merged_df = merged_df.dropna(subset=["report"]).reset_index(drop=True)

def get_path(row):
    subject_id = int(row["subject_id"])
    study_id = int(row["study_id"])
    prefix = "p10" if subject_id < 11000000 else "p11"
    rel = f"{prefix}/p{subject_id}/s{study_id}/{row['dicom_id']}.jpg"
    return file_index.get(rel, None)

merged_df["image_path"] = merged_df.apply(get_path, axis=1)
merged_df = merged_df.dropna(subset=["image_path"]).reset_index(drop=True)

if reports_zip_path:
    print("Extracting and parsing reports from mimic-cxr-reports.zip archive...", flush=True)
    study_map = {}
    for idx, row in merged_df.iterrows():
        study_map[(int(row["subject_id"]), int(row["study_id"]))] = idx
        
    reports_dict = {}
    with zipfile.ZipFile(reports_zip_path) as z:
        for name in z.namelist():
            if name.endswith(".txt"):
                parts = name.split("/")
                if len(parts) >= 4:
                    try:
                        sub_id = int(parts[2][1:])
                        std_id = int(parts[3][1:-4])
                        if (sub_id, std_id) in study_map:
                            with zipfile.ZipFile(reports_zip_path).open(name) as f:
                                text = f.read().decode("utf-8", errors="ignore")
                                lower_text = text.lower()
                                imp_idx = lower_text.find("impression:")
                                if imp_idx != -1:
                                    imp = text[imp_idx + len("impression:"):].strip()
                                else:
                                    imp = text.strip()
                                reports_dict[study_map[(sub_id, std_id)]] = " ".join(imp.split())
                    except Exception:
                        pass
                        
    replaced_count = 0
    for idx, text in reports_dict.items():
        if text:
            merged_df.at[idx, "report"] = text
            replaced_count += 1
    print(f"Successfully enriched {replaced_count} reports from ZIP archive!", flush=True)

# -------------------------------------------------------------------------
# Step C: Patient-Level Zero-Leakage GroupShuffleSplit
# -------------------------------------------------------------------------
gss_train = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(gss_train.split(merged_df, groups=merged_df["subject_id"]))

train_df = merged_df.iloc[train_idx].copy().reset_index(drop=True)
temp_df = merged_df.iloc[temp_idx].copy().reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df["subject_id"]))

val_df = temp_df.iloc[val_idx].copy().reset_index(drop=True)
test_df = temp_df.iloc[test_idx].copy().reset_index(drop=True)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

# Critical: create orig_split_idx for exact 1-to-1 memory map alignment
train_df["orig_split_idx"] = np.arange(len(train_df))
val_df["orig_split_idx"] = np.arange(len(val_df))
test_df["orig_split_idx"] = np.arange(len(test_df))

print(f"Manifest Ready: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}", flush=True)

# -------------------------------------------------------------------------
# Step D: Model 7 Breakthrough: 2D Spatial Pooling RAD-DINO Feature Extraction
#    Preserves 2D Anatomical Layout: 37x37 -> 16 Rows x 8 Cols = 128 Tokens
# -------------------------------------------------------------------------
train_feat_path = "/kaggle/working/features_spatial_train.dat"
GRID_H, GRID_W = 16, 8  # 16 rows (apical to basilar) x 8 cols (lateral to medial) = 128 tokens
print(f"Configuring 2D Spatial Adaptive Pooling: ({GRID_H} rows x {GRID_W} cols = {GRID_H * GRID_W} tokens)", flush=True)

if not os.path.exists(train_feat_path):
    print("Loading Microsoft RAD-DINO extractor from Hugging Face...", flush=True)
    class RadDino2DExtractor(nn.Module):
        def __init__(self, grid_h=16, grid_w=8):
            super().__init__()
            self.model = AutoModel.from_pretrained("microsoft/rad-dino", trust_remote_code=True)
            self.model.eval()
            self.grid_h = grid_h
            self.grid_w = grid_w
            
        @torch.no_grad()
        def forward(self, x):
            with torch.amp.autocast("cuda"):
                outputs = self.model(x)
                hidden = outputs.last_hidden_state
                cls_token = hidden[:, 0, :]  # [B, 768]
                patch_tokens = hidden[:, 1:, :]  # [B, 1369, 768]
                
                # --- 2D SPATIAL POOLING (Preserves anatomical topology) ---
                B = patch_tokens.size(0)
                # ViT patches: 37x37 grid = 1369 tokens (ordered row-major, top-left to bottom-right)
                patch_2d = patch_tokens.view(B, 37, 37, 768).permute(0, 3, 1, 2)  # [B, 768, 37, 37]
                
                # 2D Adaptive Average Pooling to (GRID_H, GRID_W):
                # 16 vertical bins capture apical, upper, mid, lower, and basilar zones
                # 8 horizontal bins capture right lateral, right lung, mediastinum, left lung, left lateral
                spatial_2d = F.adaptive_avg_pool2d(patch_2d, (self.grid_h, self.grid_w))  # [B, 768, 16, 8]
                
                # Flatten back to sequence: [B, 16, 8, 768] -> [B, 128, 768]
                spatial_128 = spatial_2d.permute(0, 2, 3, 1).reshape(B, self.grid_h * self.grid_w, 768)
                
            return spatial_128, cls_token

    raddino_extractor = RadDino2DExtractor(grid_h=GRID_H, grid_w=GRID_W).to(device)

    # RAD-DINO native 518x518 transform (14x14 patch -> 37x37=1369 tokens -> 2D pooled to 128)
    rad_transform = transforms.Compose([
        transforms.Resize(518, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(518),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5307, 0.5307, 0.5307], std=[0.2583, 0.2583, 0.2583])
    ])

    class FeatureExtractionDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df
            self.transform = transform
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            path = self.df.iloc[idx]["image_path"]
            try:
                img = Image.open(path).convert("RGB")
                tensor = self.transform(img)
            except Exception:
                tensor = torch.zeros(3, 518, 518)
            return tensor

    def extract_features_to_memmap(df_split, split_name, batch_size=32):
        n_samples = len(df_split)
        spatial_file = f"/kaggle/working/features_spatial_{split_name}.dat"
        mean_file = f"/kaggle/working/features_mean_{split_name}.dat"
        spatial_mmap = np.memmap(spatial_file, dtype="float16", mode="w+", shape=(n_samples, 128, 768))
        mean_mmap = np.memmap(mean_file, dtype="float32", mode="w+", shape=(n_samples, 768))
        ds = FeatureExtractionDataset(df_split, rad_transform)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
        idx_cursor = 0
        print(f"Extracting 2D-pooled RAD-DINO features for split: {split_name} (N={n_samples}) in batches of {batch_size}...", flush=True)
        for batch_tensors in tqdm(loader, desc=f"RAD-DINO 2D Features [{split_name}]"):
            b_size = batch_tensors.size(0)
            batch_tensors = batch_tensors.to(device)
            spatial_tokens, mean_feats = raddino_extractor(batch_tensors)
            spatial_mmap[idx_cursor:idx_cursor + b_size] = spatial_tokens.cpu().to(torch.float16).numpy()
            mean_mmap[idx_cursor:idx_cursor + b_size] = mean_feats.cpu().to(torch.float32).numpy()
            idx_cursor += b_size
        spatial_mmap.flush()
        mean_mmap.flush()
        print(f"Split {split_name} extraction complete: {n_samples} samples saved.", flush=True)

    extract_features_to_memmap(train_df, "train", batch_size=32)
    extract_features_to_memmap(val_df, "val", batch_size=32)
    extract_features_to_memmap(test_df, "test", batch_size=32)

    del raddino_extractor
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("Precomputed 2D RAD-DINO features already exist on disk.", flush=True)

# -------------------------------------------------------------------------
# Step E: Patient-Leakage-Free FAISS Index & RAG Pre-calculation
# -------------------------------------------------------------------------
enriched_manifest_path = "/kaggle/working/mimic_master_split_with_rag.csv"
print("Building FAISS FlatIP Index...", flush=True)
train_mean_mmap = np.memmap("/kaggle/working/features_mean_train.dat", dtype="float32", mode="r", shape=(len(train_df), 768))
train_vectors = np.array(train_mean_mmap)
faiss.normalize_L2(train_vectors)
dim = 768
index = faiss.IndexFlatIP(dim)
index.add(train_vectors)
print(f"FAISS Index Ready with {index.ntotal} training exemplar vectors.", flush=True)

def compute_rag_retrievals(split_df, mean_mmap_file, is_train=False):
    mean_mmap = np.memmap(mean_mmap_file, dtype="float32", mode="r", shape=(len(split_df), 768))
    query_vectors = np.array(mean_mmap)
    faiss.normalize_L2(query_vectors)
    k_search = 50 if is_train else 10
    D, I = index.search(query_vectors, k_search)
    retrieved_reports = []
    for i in range(len(split_df)):
        curr_subj = split_df.iloc[i]["subject_id"]
        chosen_idx = None
        if is_train:
            # Strict guarantee: exemplar must NOT come from the same patient
            for neighbor_idx in I[i]:
                cand_subj = train_df.iloc[neighbor_idx]["subject_id"]
                if cand_subj != curr_subj:
                    chosen_idx = neighbor_idx
                    break
            if chosen_idx is None:
                chosen_idx = I[i, 1]
        else:
            chosen_idx = I[i, 0]
            
        raw_rep = train_df.iloc[chosen_idx]["report"]
        if pd.isna(raw_rep) or str(raw_rep).strip().lower() in ["", "nan", "none"]:
            clean_rep = "Normal chest radiograph with no acute cardiopulmonary abnormality."
        else:
            clean_rep = str(raw_rep).strip()
        retrieved_reports.append(clean_rep)
        
    split_df = split_df.copy()
    split_df["retrieved_report"] = retrieved_reports
    return split_df

train_df = compute_rag_retrievals(train_df, "/kaggle/working/features_mean_train.dat", is_train=True)
val_df = compute_rag_retrievals(val_df, "/kaggle/working/features_mean_val.dat", is_train=False)
test_df = compute_rag_retrievals(test_df, "/kaggle/working/features_mean_test.dat", is_train=False)

master_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
master_df.to_csv(enriched_manifest_path, index=False)
print(f"Saved RAG-enriched master manifest to {enriched_manifest_path}", flush=True)

# -------------------------------------------------------------------------
# Step F: Final Verification of Exported Artifacts
# -------------------------------------------------------------------------
print("\n" + "="*70, flush=True)
print("MODEL 7 DATA PREP COMPLETE: VERIFYING ARTIFACTS IN /kaggle/working", flush=True)
print("="*70, flush=True)
expected_files = [
    "mimic_master_split_with_rag.csv",
    "features_spatial_train.dat", "features_spatial_val.dat", "features_spatial_test.dat",
    "features_mean_train.dat", "features_mean_val.dat", "features_mean_test.dat"
]

for ef in expected_files:
    full = os.path.join("/kaggle/working", ef)
    if os.path.exists(full):
        sz_mb = os.path.getsize(full) / (1024 * 1024)
        print(f"  [OK] {ef:<35} ({sz_mb:.1f} MB)", flush=True)
    else:
        print(f"  [MISSING] {ef}", flush=True)

print("="*70, flush=True)
print("NEXT STEP: Create a new Kaggle Dataset from these outputs (e.g. 'mimic-cxr-model-7-prep-data')", flush=True)
print("and attach it to your Model 7 Training Notebook!", flush=True)
